# BERT Approach

This notebook presents the bert based approach in solving the three way classification problem of clarity labeling

In [11]:
from transformers import BertTokenizer, BertForSequenceClassification
import torch
from torch.utils.data import DataLoader, TensorDataset
import pandas as pd
from torch.optim import AdamW

In [12]:
df = pd.read_csv('/content/dataset/training_data_processed.csv')
drop_cols = ['evasion_label', 'Unnamed: 0', 'affirmative_questions']
df = df.drop(columns= drop_cols)
df.head()

,question_order,interview_question,interview_answer,gpt3.5_summary,gpt3.5_prediction,question,clarity_label
0,1,Q. Of the Biden administration. And accused th...,"Well, look, first of all, theI am sincere abou...",The question consists of 2 parts: \n1. How wou...,Question part: 1. How would you respond to the...,How would you respond to the accusation that t...,Clear Reply
1,1,Q. Of the Biden administration. And accused th...,"Well, look, first of all, theI am sincere abou...",The question consists of 2 parts: \n1. How wou...,Question part: 1. How would you respond to the...,Do you think President Xi is being sincere abo...,Ambivalent
2,2,Q. No worries. Do you believe the country's sl...,"Look, I think China has a difficult economic p...",The question consists of two parts:\n\n1. Q1: ...,Question part: Q1 - Do you believe the country...,Do you believe the country's slowdown and gro...,Ambivalent
3,2,Q. No worries. Do you believe the country's sl...,"Look, I think China has a difficult economic p...",The question consists of two parts:\n\n1. Q1: ...,Question part: Q1 - Do you believe the country...,Are you worried about the meeting between Pre...,Ambivalent
4,3,"Q. I can imagine. It is evening, I'd like to r...","Well, I hope I get to see Mr. Xi sooner than l...",The question consists of 3 parts:\n1. Is the P...,Question part: 1. Is the President's engagemen...,Is the President's engagement with Asian coun...,Clear Reply


In [13]:
label_map = {
    'Clear Reply': 0,
    'Clear Non-Reply': 1,
    'Ambivalent': 2
}
num_labels = len(label_map)
model_name = 'bert-base-uncased'


model = BertForSequenceClassification.from_pretrained(model_name, num_labels=num_labels) # For binary classification
tokenizer = BertTokenizer.from_pretrained(model_name)


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [14]:
def data_prep(questions, answers, clarity_label):
    encodings = tokenizer(questions.tolist(),
                          answers.tolist(),
                          padding= True,
                          truncation = True,
                          max_length = 512,
                          return_tensors = 'pt'
                          )
    label_tensors = torch.tensor([label_map[l] for l in clarity_label.tolist()])
    dataset = TensorDataset(
        encodings['input_ids'],
        encodings['attention_mask'],
        encodings['token_type_ids'],
        label_tensors
        )
    return dataset


In [15]:

from sklearn.model_selection import train_test_split

# Assuming 'df' is your full DataFrame
Q_train, Q_val, A_train, A_val, L_train, L_val = train_test_split(
    df['Interview_question'],
    df['interview_answer'],
    df['clarity_label'],
    test_size=0.2,
    random_state=40
    )



BATCH_SIZE = 16

train_dataset = data_prep(Q_train, A_train, L_train)
val_dataset = data_prep(Q_val, A_val, L_val)


train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pai

### Model Training

With preprocessing done we can now gop on to build the training loop to finetune the BERT model

In [17]:
import numpy as np


def eval_model(model, data_loader, device):
  model.eval()

  correct_predictions = 0
  total_predictions = 0


  with torch.no_grad():
    for batch in data_loader:
      input_ids, attention_mask, token_type_ids, labels = [t.to(device) for t in batch]

      outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                token_type_ids=token_type_ids
                )

      _, predicted = torch.max(outputs.logits, 1)

      total_predictions += labels.size(0)
      correct_predictions += (predicted == labels).sum().item()
  model.train()

  accuracy = 100 * correct_predictions / total_predictions
  return accuracy

In [18]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
epochs = 20
optimizer = AdamW(model.parameters(), lr = 5e-5)



for epoch in range(epochs):
    print(f"--- Starting Epoch {epoch+1}/{epochs} ---")
    model.train()
    total_train_loss = 0
    for batch in train_loader:
        input_ids = batch[0].to(device)
        attention_mask = batch[1].to(device)
        token_type_ids = batch[2].to(device)
        labels = batch[3].to(device)

        model.zero_grad()

        outputs = model(input_ids= input_ids,
                        attention_mask = attention_mask,
                        token_type_ids = token_type_ids,
                        labels = labels
                        )

        loss = outputs.loss
        logits = outputs.logits

        loss.backward()
        optimizer.step()
        total_train_loss += loss.item()
    avg_train_loss = total_train_loss / len(train_loader)

    # Run evaluation function
    val_accuracy = eval_model(model, val_loader, device)

    print(f"Epoch {epoch+1} Summary:")
    print(f"  Training Loss: {avg_train_loss:.4f}")
    print(f"  Validation Accuracy: {val_accuracy:.2f}%")

--- Starting Epoch 1/5 ---
Epoch 1 Summary:
  Training Loss: 0.8381
  Validation Accuracy: 64.64%
--- Starting Epoch 2/5 ---
Epoch 2 Summary:
  Training Loss: 0.6959
  Validation Accuracy: 60.43%
--- Starting Epoch 3/5 ---
Epoch 3 Summary:
  Training Loss: 0.4972
  Validation Accuracy: 64.06%
--- Starting Epoch 4/5 ---
Epoch 4 Summary:
  Training Loss: 0.3205
  Validation Accuracy: 64.64%
--- Starting Epoch 5/5 ---
Epoch 5 Summary:
  Training Loss: 0.1737
  Validation Accuracy: 67.83%
